In [1]:
import numpy as np
import pandas as pd

### Day 1 — Bird Diversity & Habitat Analysis

In [ ]:
np.random.seed(42)

sites = ["East Kolkata Wetlands", "Sundarban Fringe", "Nalban", "Santragachi"]
species = ["Egret", "Kingfisher", "Heron", "Cormorant", "Ibis", "Pied Hornbill"]

dates = pd.date_range("2025-01-01", "2025-06-01", freq="MS")

rows = []

for date in dates:
    for site in sites:
        for sp in species:
            rows.append({
                "date": date,
                "site": site,
                "species": sp,
                "count": np.random.poisson(
                    np.random.uniform(5, 30)
                ),
                "observation_hours": np.random.uniform(2, 8)
            })

birds = pd.DataFrame(rows)

# Introduce some missing observations
missing_idx = np.random.choice(
    birds.index,
    size=12,
    replace=False
)

birds.loc[missing_idx, "count"] = np.nan

birds.head()

In [ ]:
birds.info()
birds.describe()

In [ ]:
birds['count'].unique()
birds[birds['count'].isna()]
birds.fillna(birds['count'].mean(), inplace=True)

In [ ]:
birds['month'] = birds['date'].dt.month
birds['day_name'] = birds['date'].dt.day_name()
birds['density'] = birds['count'] / birds['observation_hours']
birds.head()

In [ ]:
birds[birds['count']>birds['count'].mean()]
birds.query('species == "Egret" | species =="Kingfisher" | species == "Heron" | species == "Cormorant" | species == "Ibis"').where(birds['count'] > birds['count'].mean()).dropna()
birds.query('species == "Kingfisher"').where(birds['site'] == 'Nalban').dropna().sort_values(by='date',ascending=False)

In [ ]:
birds.where(birds['observation_hours']>=birds['observation_hours'].quantile(0.9)).dropna()
birds['observation_hours'].quantile([0.25,0.5,0.75,0.9])

In [ ]:
birds.groupby(['site','species'])['count'].agg(['min','max','mean','sum']).sort_values(by=['site','sum'] ,ascending=[True,False] )
birds.groupby('species')['count'].agg(['max','min','std'])
birds.groupby(['site','month'])['count'].agg(['sum']).sort_values(by=['site','sum'],ascending=[True,False])

In [ ]:
birds_pivot = pd.pivot_table(birds, index='site', columns='species',values='count',aggfunc='sum')
birds_pivot

In [ ]:
ovser_pivot = pd.pivot_table(birds,index='site',columns='species',values='observation_hours',aggfunc='mean')
ovser_pivot

### Day 2 — Water Quality & Pollution Analysis

In [21]:
import numpy as np
import pandas as pd

np.random.seed(24)

stations = ["Hooghly_A", "Hooghly_B", "Damodar_A", "Damodar_B", "Rupnarayan"]
months = pd.date_range("2025-01-01", "2025-08-01", freq="MS")

rows = []

for month in months:
    for station in stations:
        rows.append({
            "date": month,
            "station": station,
            "temperature": np.random.normal(27, 3),
            "dissolved_oxygen": np.random.normal(6, 1.2),
            "bod": np.random.normal(4, 1.5),
            "nitrate": np.random.normal(3.5, 1.5),
            "sampling_depth": np.random.uniform(0.5, 3.0)
        })

water = pd.DataFrame(rows)

# Introduce realistic missing measurements
missing_idx = np.random.choice(
    water.index,
    size=10,
    replace=False
)

water.loc[missing_idx, "dissolved_oxygen"] = np.nan
water.loc[
    np.random.choice(water.index, 6, replace=False),
    "nitrate"
] = np.nan

water.head()

,date,station,temperature,dissolved_oxygen,bod,nitrate,sampling_depth
0,2025-01-01,Hooghly_A,30.987637,5.075960,3.525579,2.013784,1.301298
1,2025-01-01,Hooghly_B,31.229921,NaN,4.119309,4.899379,1.118234
2,2025-01-01,Damodar_A,29.036414,8.267127,5.442308,NaN,2.606949
3,2025-01-01,Damodar_B,29.927131,NaN,5.889212,5.843651,1.069208
4,2025-01-01,Rupnarayan,31.178564,NaN,4.182503,5.311404,2.394495


In [ ]:
water.info()
water.describe()

In [ ]:
# finding null values and replace them with mean
water[water['dissolved_oxygen'].isna()]
water.fillna(value={
    'dissolved_oxygen': water['dissolved_oxygen'].mean(),
    'bod' : water['bod'].mean(),
    'nitrate' : water['nitrate'].mean()
},inplace=True)

In [33]:
# seperating months in a new column
water['month'] = water['date'].dt.month

In [50]:
water.groupby('month')[['dissolved_oxygen','bod','nitrate']].agg(['min','max','mean'])

dissolved_oxygen                           bod                      \
                   min       max      mean       min       max      mean   
month                                                                      
1             5.075960  8.267127  6.284384  3.525579  5.889212  4.631782   
2             4.408844  6.348042  5.614807  1.044567  6.143476  4.416136   
3             3.741178  7.117565  5.186892  3.500588  6.284867  4.754373   
4             4.556828  8.377701  6.075642  1.554702  6.992887  4.039748   
5             5.246975  7.476893  6.370299  2.321859  6.863098  4.261740   
6             4.034295  7.384201  5.908554  1.273385  4.651945  3.240017   
7             5.562161  7.331687  6.558538  2.467843  4.711069  3.748570   
8             5.966585  6.676326  6.211101  2.877510  6.128184  4.720302   

        nitrate                      
            min       max      mean  
month                                
1      2.013784  5.843651  4.325902  
2      0.365969  4.705859  3.420544  
3      1.896237  6.839239  4.009155  
4     -0.479793  4.992482  2.956545  
5      1.716544  6.563304  4.120937  
6      1.000639  4.945652  3.309752  
7      1.263398  3.698315  2.621585  
8      1.335221  4.918929  3.725929